# Day 01 — Environment Setup + pandas Warmup
**Estimated time:** 45–60 min  
**Dataset:** `materials_inventory.csv`

## Learning Objectives
- Confirm the analytics stack is functional (numpy, pandas, matplotlib)
- Load a real SAP materials dataset and perform structured first-pass inspection
- Fix raw dtypes and derive initial business insights through exploration


In [ ]:
# ── 1. Environment verification ──────────────────────────────────────────────
import sys, importlib

required = ["numpy", "pandas", "matplotlib"]
for pkg in required:
    mod = importlib.import_module(pkg)
    print(f"✓ {pkg} {mod.__version__}")

print(f"Python {sys.version}")


In [ ]:
# ── 2. Load materials_inventory.csv ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Path assumes notebook lives in week_01_02/ and datasets are one level up
DATA_DIR = "../datasets"
df = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")

print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)


In [ ]:
# ── 3. First-pass inspection ─────────────────────────────────────────────────
display(df.head(10))


In [ ]:
%%time
# ── 4. Descriptive statistics ────────────────────────────────────────────────
display(df.describe(include="all").T)


In [ ]:
# ── 5. Fix data types ────────────────────────────────────────────────────────
# LAST_MOVEMENT_DATE → datetime
df["LAST_MOVEMENT_DATE"] = pd.to_datetime(df["LAST_MOVEMENT_DATE"], errors="coerce")

# Numeric fields — coerce stray strings
for col in ["LABST", "STPRS"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# MATNR, WERKS as strings (leading-zero safe)
for col in ["MATNR", "WERKS"]:
    df[col] = df[col].astype(str).str.zfill({"MATNR": 18, "WERKS": 4}[col])

print("Fixed dtypes:")
print(df[["MATNR","WERKS","LABST","STPRS","LAST_MOVEMENT_DATE"]].dtypes)
print(f"\nNull counts:\n{df.isnull().sum()}")


In [ ]:
# ── 6. Exploratory Task A: Unique plants ─────────────────────────────────────
unique_plants = df["WERKS"].nunique()
print(f"Distinct plants (WERKS): {unique_plants}")
print(df["WERKS"].value_counts().rename_axis("WERKS").reset_index(name="record_count"))


In [ ]:
# ── 7. Exploratory Task B: Material type distribution ────────────────────────
mat_type_dist = (df.groupby("MATERIAL_TYPE")["MATNR"]
                   .nunique()
                   .sort_values(ascending=False)
                   .rename("distinct_materials")
                   .reset_index())
print(mat_type_dist)


In [ ]:
%%time
# ── 8. Exploratory Task C: Top 20 materials by inventory value ────────────────
# Inventory value = stock quantity × standard price
df["INV_VALUE"] = df["LABST"] * df["STPRS"]

top20 = (df.sort_values("INV_VALUE", ascending=False)
           .head(20)[["MATNR","MAKTX","WERKS","LABST","STPRS","INV_VALUE"]]
           .reset_index(drop=True))
display(top20)


In [ ]:
# ── Quick bar chart: inventory value by material type ────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
df.groupby("MATERIAL_TYPE")["INV_VALUE"].sum().sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Total Inventory Value")
ax.set_title("Inventory Value by Material Type")
ax.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


---
## Daily Prompt

**Find all materials with zero stock (`LABST == 0`) but a movement date in the last 30 days. What does this suggest from an SAP inventory management perspective?**

Consider:
- Are these goods-receipt items that were immediately consumed?
- Could they be transfer orders with a timing lag?
- What MRP types are represented in this set — does that tell you anything?

```python
# YOUR SOLUTION
import datetime

cutoff = pd.Timestamp.today() - pd.Timedelta(days=30)

# Filter materials: zero stock AND recent movement
result = df.loc[
    (df["LABST"] == 0) &
    (df["LAST_MOVEMENT_DATE"] >= cutoff)
]

print(f"Materials found: {len(result)}")
# YOUR SOLUTION: explore MRPTYPE, MATERIAL_TYPE distribution in result
```

> **Hint:** Group `result` by `MRPTYPE` and `MATERIAL_TYPE`. High counts of `PD` (MRP-planned) suggest make-to-order or just-in-time consumption. High counts of `ND` (no MRP) with recent movement could indicate manual adjustments or write-offs.
